# Perplexity Distribution From Result JSON
Load evaluation result JSON and plot sample-level perplexity distribution, split by correctness.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')

In [ ]:
# Update this path to any result JSON file
RESULT_JSON_PATH = Path('Results/spare/eval_gemma_refusal_open_coeff_15_20260410_220127.json')

In [ ]:
with RESULT_JSON_PATH.open('r', encoding='utf-8') as f:
    payload = json.load(f)

result = payload.get('result', {}) or {}
samples = result.get('samples', []) or []

rows = []
for i, s in enumerate(samples):
    meta = s.get('metadata', {}) or {}
    ppl = meta.get('perplexity', None)
    if ppl is None:
        continue
    rows.append({
        'sample_idx': i,
        'perplexity': float(ppl),
        'is_correct': int(s.get('is_correct', 0)),
    })

df = pd.DataFrame(rows)
if df.empty:
    raise ValueError('No sample-level metadata.perplexity found in this result file.')

display(df.head())
print(f'n_samples_with_perplexity = {len(df)}')
print(f'overall_result_perplexity = {result.get("perplexity", None)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    data=df,
    x='perplexity',
    hue='is_correct',
    bins=20,
    alpha=0.5,
    element='step',
    ax=axes[0],
)
axes[0].set_title('Perplexity Histogram (sample-level)')
axes[0].set_xlabel('perplexity')
axes[0].set_ylabel('count')

sns.boxplot(
    data=df,
    x='is_correct',
    y='perplexity',
    hue='is_correct',
    dodge=False,
    ax=axes[1],
)
if axes[1].legend_ is not None:
    axes[1].legend_.remove()
axes[1].set_title('Perplexity by Correctness')
axes[1].set_xlabel('is_correct (0=incorrect, 1=correct)')
axes[1].set_ylabel('perplexity')

plt.tight_layout()
plt.show()